## Common Challenges When Handling PDFs

- **Text extraction:** PDFs may contain scanned images, embedded fonts, or complex layouts that make text extraction difficult.
- **Optical character recognition (OCR):** Scanned PDFs require OCR, which can introduce spelling and formatting errors.
- **Preserving formatting:** Tables, columns, images, headers, and footers may not retain their original structure after extraction.
- **Large file sizes:** High-resolution images and embedded content can make PDFs difficult to store, upload, or process.
- **Encrypted or protected files:** Passwords, permissions, and digital rights management can restrict access or editing.
- **Inconsistent document structure:** PDFs from different sources may use varying layouts, metadata, and encoding methods.
- **Table extraction:** Complex or irregular tables are often challenging to convert into structured data.
- **Missing metadata:** Author, title, creation date, and other metadata may be incomplete or inaccurate.
- **Corrupted files:** Damaged PDFs may fail to open or produce incomplete extraction results.
- **Accessibility issues:** PDFs may lack searchable text, proper headings, alternative text, or screen-reader support.
- **Security risks:** PDFs can contain malicious scripts, embedded files, or harmful links.
- **Version compatibility:** Features supported by newer PDF standards may not work correctly in older software.

In [1]:
## 

raw_text = """Company Financial Report


The financial performance for fiscal year 2025 
shows significant growth in profitability.


Revenue increased by 25%.

The company's efficiency improved due to workflow optimization.

page 1 of 10 """

# appluy cleaning funciton

def clean_text(text):
    text = " ".join(text.split())

    # Fix ligatures
    text = text.replace("ﬁ", "fi")
    text = text.replace("ﬂ", "fl")
    text = text.replace("ﬀ", "ff")

    return text

cleaned_text = clean_text(raw_text)

print(f"BEFORE")
print(repr(raw_text[:100]))
print("\nAFTER")
print(repr(cleaned_text[:100]))


BEFORE
'Company Financial Report\n\n\nThe financial performance for fiscal year 2025 \nshows significant growth '

AFTER
'Company Financial Report The financial performance for fiscal year 2025 shows significant growth in '


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
## PDf parsing

from pathlib import Path
import os

# Move to the project root, regardless of current notebook location
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: /Users/bk/Downloads/Code/udemy/RAG-Projects/RAG-Bootcamp


In [17]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List, Dict

class SmartPDFProcessor:
    """Advance PDf processing with error handling"""
    def __init__(self,chunk_size=1000, chunk_overlap=100):
        self.chunk_size=chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap = self.chunk_overlap,
            separators = [" "]
        )

    def _clean_text(self, text:str) -> str:
        """Clean extracted text"""
        text = " ".join(text.split())
        # Fix ligatures
        text = text.replace("ﬁ", "fi")
        text = text.replace("ﬂ", "fl")
        text = text.replace("ﬀ", "ff")

        return text
    
    def process_pdf(self, pdf_path:str) -> List[Document]:
        """Process PDF with smart chunking and metadata"""

        #Load PDf
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()

        #process each page
        processed_chunks = []
        for page_num, page in enumerate(pages):
            #Clean text
            cleaned_text = self._clean_text(page.page_content)

            #Skip nearly empty pages
            if len(cleaned_text.strip()) < 20:
                continue

            #create chunk with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[
                    {
                        **page.metadata,
                        "page":page_num+1,
                        "total_pages":len(pages),
                        "chunk_method":"inbuilt_smart_pdf_processor",
                        "charact_count":len(cleaned_text)
                    }
                ]
            )
            processed_chunks.extend(chunks)

        return processed_chunks

In [18]:
preprocessor = SmartPDFProcessor()

In [20]:
try:
    smart_chunks=preprocessor.process_pdf("data/raw/pdf/1706.03762v7.pdf") 
    print(f"Processed into {len(smart_chunks)} smart chunks")

    if smart_chunks:
        print("\nSample chunk metadata: ")
        for key, value in smart_chunks[0].metadata.items():
            print(f" {key} : {value}")
except Exception as e:
    print(f"Processing error {e}")

Processed into 49 smart chunks

Sample chunk metadata: 
 producer : pdfTeX-1.40.25
 creator : LaTeX with hyperref
 creationdate : 2024-04-10T21:11:43+00:00
 author : 
 keywords : 
 moddate : 2024-04-10T21:11:43+00:00
 ptex.fullbanner : This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
 subject : 
 title : 
 trapped : /False
 source : data/raw/pdf/1706.03762v7.pdf
 total_pages : 15
 page : 1
 page_label : 1
 chunk_method : inbuilt_smart_pdf_processor
 charact_count : 2855
